<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">Comprehensive Multimodal Training Guide</h1>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">This notebook provides a comprehensive guide to multimodal training using LLaMA-Factory, covering:</p>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Image-Text Training</strong>: Vision-language model fine-tuning</li>
<li style="margin:6px 0;"><strong>Audio Processing</strong>: Speech and audio understanding</li>
<li style="margin:6px 0;"><strong>Video Processing</strong>: Video content analysis</li>
<li style="margin:6px 0;"><strong>Dataset Preparation</strong>: Multimodal data formatting</li>
<li style="margin:6px 0;"><strong>Model Training</strong>: Configuration for different modalities</li>
<li style="margin:6px 0;"><strong>Evaluation</strong>: Multimodal model assessment</li>
</ol>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Table of Contents</h2>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#setup-and-installation">Setup and Installation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#multimodal-dataset-preparation">Multimodal Dataset Preparation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#image-text-training">Image-Text Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#audio-processing">Audio Processing</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#video-analysis">Video Analysis</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#training-configurations">Training Configurations</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#evaluation-and-testing">Evaluation and Testing</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#best-practices">Best Practices</a></li>
</ul>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Setup and Installation</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">First, let's install the required dependencies for multimodal processing.</p>
</div>


In [ ]:
# Install multimodal dependencies
%pip install -r requirements.txt
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers[torch] datasets pillow opencv-python librosa moviepy
%pip install timm ftfy regex
%pip install av  # For audio/video processing

# Import required libraries
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForVision2Seq, AutoProcessor
from peft import PeftModel
from llamafactory import ChatModel
import json
import os
import yaml
from PIL import Image
import requests
import librosa
import cv2
import numpy as np

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Multimodal Dataset Preparation</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">Let's prepare datasets for different multimodal training scenarios.</p>
</div>


In [ ]:
# 1. Image-Text Dataset (Vision-Language)
image_text_data = [
    {
        "messages": [
            {"role": "user", "content": "Describe this image in detail."},
            {"role": "assistant", "content": "This is an image that shows..."}
        ],
        "images": ["path/to/image1.jpg"]
    },
    {
        "messages": [
            {"role": "user", "content": "What's in this picture?"},
            {"role": "assistant", "content": "The image contains..."}
        ],
        "images": ["path/to/image2.jpg"]
    }
]

# Save image-text dataset
with open('data/mllm_demo.json', 'w') as f:
    json.dump(image_text_data, f, indent=2)


In [ ]:
# 2. Audio-Text Dataset (Speech Understanding)
audio_text_data = [
    {
        "messages": [
            {"role": "user", "content": "Transcribe this audio and explain what it's saying."},
            {"role": "assistant", "content": "This audio contains..."}
        ],
        "audios": ["path/to/audio1.wav"]
    },
    {
        "messages": [
            {"role": "user", "content": "What language is being spoken in this recording?"},
            {"role": "assistant", "content": "The audio is in..."}
        ],
        "audios": ["path/to/audio2.mp3"]
    }
]

# Save audio-text dataset
with open('data/mllm_audio_demo.json', 'w') as f:
    json.dump(audio_text_data, f, indent=2)


In [ ]:
# 3. Video-Text Dataset (Video Analysis)
video_text_data = [
    {
        "messages": [
            {"role": "user", "content": "Describe what happens in this video."},
            {"role": "assistant", "content": "This video shows..."}
        ],
        "videos": ["path/to/video1.mp4"]
    },
    {
        "messages": [
            {"role": "user", "content": "What activities are taking place in this footage?"},
            {"role": "assistant", "content": "The video captures..."}
        ],
        "videos": ["path/to/video2.avi"]
    }
]

# Save video-text dataset
with open('data/mllm_video_demo.json', 'w') as f:
    json.dump(video_text_data, f, indent=2)
